In [1]:
# Project setup
import sys
import os

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate

from src.data.loaders import PointsDataset
from src.features.transformations import cartesian_to_polar, polar_to_curvilinear_r2cos2theta

In [3]:
current_dir = Path.cwd()

# Load the dataset
data_path = current_dir.parent / "data" / "raw" / "points.json"
dataset = PointsDataset.from_json(data_path=data_path)

## Data Preparation

In [4]:
# Transform data points into polar form
polar_dataset = dataset.transform(cartesian_to_polar)
# Transform data points from polar into curvilinear form
cos_sqd_dataset = polar_dataset.transform(polar_to_curvilinear_r2cos2theta)

# Model Comparison Framework

5-fold cross-validation

In [5]:
# Initialize with 5 folds
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

folds = list(skf.split(dataset.X, dataset.y))

## Logistic Regression on Polar Coordinates

In [6]:
logreg_pipeline = Pipeline([
    ("logreg", LogisticRegression())
])

scoring = ["accuracy", "precision", "recall", "f1"]

logreg_results = cross_validate(
    logreg_pipeline,
    polar_dataset.X,
    polar_dataset.y,
    cv=folds,
    scoring=scoring,
    return_train_score=True
)

## Logistic Regression on Featured Curvilinear Coordinates

In [7]:
logreg_cos2_pipeline = Pipeline([
    ("logreg_cos2", LogisticRegression())
])

scoring = ["accuracy", "precision", "recall", "f1"]

logreg__cos2_results = cross_validate(
    logreg_cos2_pipeline,
    cos_sqd_dataset.X,
    cos_sqd_dataset.y,
    cv=folds,
    scoring=scoring,
    return_train_score=True
)

## Support Vector Machines

In [8]:
svm_pipeline = Pipeline([
    ("svm", SVC(kernel="rbf"))
])

scoring = ["accuracy", "precision", "recall", "f1"]

svm_results = cross_validate(
    svm_pipeline,
    dataset.X,
    dataset.y,
    cv=folds,
    scoring=scoring,
    return_train_score=True
)

## Custom Parametric Model

## Cross-Validation Results

In [9]:
models_test = {}
models_train = {}

for key in svm_results.keys():
    svm_metrics = svm_results[key]
    logreg_metrics = logreg_results[key]
    logreg_cos2_metrics = logreg__cos2_results[key]

    if 'train' in key:
        models_train[key] = {}
        my_dict = models_train
    else:
        models_test[key] = {}
        my_dict = models_test

    my_dict[key] = [f'{np.mean(svm_metrics):.5f} +/- {np.std(svm_metrics):.5f}', 
                    f'{np.mean(logreg_metrics):.5f} +/- {np.std(logreg_metrics):.5f}', 
                    f'{np.mean(logreg_cos2_metrics):.5f} +/- {np.std(logreg_cos2_metrics):.5f}']

model_names = ['SVM', 'LogReg', 'LogReg-cos2']

models_test_df = pd.DataFrame(models_test, index=model_names)
models_train_df = pd.DataFrame(models_train, index=model_names)

In [10]:
display(models_test_df)
display(models_train_df)

,fit_time,score_time,test_accuracy,test_precision,test_recall,test_f1
SVM,0.00146 +/- 0.00088,0.00454 +/- 0.00214,0.94947 +/- 0.03164,0.98182 +/- 0.03636,0.92000 +/- 0.07483,0.94720 +/- 0.03526
LogReg,0.00276 +/- 0.00279,0.00403 +/- 0.00154,0.93947 +/- 0.01976,0.94364 +/- 0.04614,0.94000 +/- 0.04899,0.93990 +/- 0.02008
LogReg-cos2,0.00213 +/- 0.00146,0.00448 +/- 0.00190,0.97000 +/- 0.04000,0.96182 +/- 0.04685,0.98000 +/- 0.04000,0.97048 +/- 0.03977


,train_accuracy,train_precision,train_recall,train_f1
SVM,0.95962 +/- 0.01468,0.97010 +/- 0.02378,0.95000 +/- 0.02236,0.95961 +/- 0.01449
LogReg,0.96468 +/- 0.01463,0.96499 +/- 0.01225,0.96500 +/- 0.02000,0.96494 +/- 0.01464
LogReg-cos2,0.96972 +/- 0.01006,0.96085 +/- 0.01180,0.98000 +/- 0.01000,0.97031 +/- 0.00985


## Statistical Tests